In [ ]:
print("\n" + "=" * 80)
print("STRATEGIC RECOMMENDATIONS FOR PRICING & SEGMENTATION")
print("=" * 80)

recommendations = {
    'h1': generate_business_interpretation(h1_result) if 'error' not in h1_result else "Unable to analyze province differences",
    'h4': generate_business_interpretation(h4_result) if 'error' not in h4_result else "Unable to analyze gender differences"
}

# For H2 and H3, use fallback if needed
if 'error' not in h2_result:
    recommendations['h2'] = generate_business_interpretation(h2_result)
    
if 'error' not in h3_result:
    recommendations['h3'] = generate_business_interpretation(h3_result)

for key, rec in recommendations.items():
    print(f"\n{rec}")

print("\n" + "=" * 80)
print("KEY TAKEAWAYS")
print("=" * 80)

rejected_count = sum(1 for r in [h1_result, h2_result, h3_result, h4_result] 
                     if 'error' not in r and r.get('significant', False))

print(f"• {rejected_count} out of 4 hypotheses were rejected at α=0.05")
print("• Rejected hypotheses indicate statistically significant differences that warrant")
print("  incorporation into the segmentation and pricing strategy")
print("• Effect sizes should also be considered - statistical significance alone may not")
print("  indicate practical importance for pricing decisions")
print("• Recommend implementing A/B testing on new pricing tiers in Q2 2026")

## Section 8: Business Recommendations

Key findings and strategic recommendations from the hypothesis testing analysis.

In [ ]:
# Compile results into summary table
results_list = []
for i, result in enumerate([h1_result, h2_result, h3_result, h4_result], 1):
    if 'error' not in result:
        results_list.append({
            'Hypothesis': result['test'],
            'KPI': 'Claim Frequency' if result['test_type'] == 'Chi-Squared' else 'Margin' if 'Margin' in result['test'] else 'Claim Frequency',
            'Test Type': result['test_type'],
            'Statistic': f"{result['statistic']:.4f}",
            'P-Value': f"{result['p_value']:.6f}",
            'Effect Size': f"{result.get('effect_size', 0):.4f}",
            'Group A': f"{result.get('group_a_mean', result.get('group_a_median', 0)):.4f}",
            'Group B': f"{result.get('group_b_mean', result.get('group_b_median', 0)):.4f}",
            'Difference': f"{result['difference']:.4f}",
            'Decision': '✅ REJECT H₀' if result['significant'] else '❌ Fail to Reject H₀'
        })

results_df = pd.DataFrame(results_list)
print("\n" + "=" * 150)
print("HYPOTHESIS TESTING SUMMARY TABLE")
print("=" * 150)
print(results_df.to_string(index=False))
print("\nSignificance Level: α = 0.05")

## Section 7: Summary Results Table

Consolidated hypothesis testing results with statistical significance assessment.

In [ ]:
print("=" * 80)
print("HYPOTHESIS 4: Gender Risk Differences")
print("=" * 80)

h4_result = analyze_gender_risk(df)
if 'error' not in h4_result:
    print(format_hypothesis_result(h4_result))
    print("\n" + "="*80)
    print("BUSINESS INTERPRETATION:")
    print("="*80)
    print(generate_business_interpretation(h4_result))
else:
    print(f"Note: {h4_result['error']}")

## Section 6: Hypothesis 4 - Gender Risk Differences

**H₀: There is no significant risk difference between Women and Men.**

Test claim frequency differences between genders using chi-squared test.

In [ ]:
print("=" * 80)
print("HYPOTHESIS 3: ZipCode Margin Differences")
print("=" * 80)

h3_result = analyze_zipcode_margin(df)
if 'error' not in h3_result:
    print(format_hypothesis_result(h3_result))
    print("\n" + "="*80)
    print("BUSINESS INTERPRETATION:")
    print("="*80)
    print(generate_business_interpretation(h3_result))
else:
    print(f"Note: {h3_result['error']}")
    # Fallback: test province margin differences
    print("\nFallback: Testing Province Margin Differences...")
    df_temp = df.copy()
    prov_counts = df_temp['Province'].value_counts()
    prov_a, prov_b = prov_counts.index[:2]
    
    group_a = df_temp[df_temp['Province'] == prov_a]['Margin']
    group_b = df_temp[df_temp['Province'] == prov_b]['Margin']
    
    h3_result = independent_t_test(group_a, group_b, test_name=f"Province Margin: {prov_a} vs {prov_b}")
    h3_result['group_a_label'] = prov_a
    h3_result['group_b_label'] = prov_b
    
    print(format_hypothesis_result(h3_result))
    print("\n" + "="*80)
    print("BUSINESS INTERPRETATION:")
    print("="*80)
    print(generate_business_interpretation(h3_result))

## Section 5: Hypothesis 3 - ZipCode Margin Differences

**H₀: There is no significant margin (profit) difference between zip codes.**

Test profit differences (Margin = TotalPremium - TotalClaims) between segments using t-test.

In [ ]:
print("=" * 80)
print("HYPOTHESIS 2: ZipCode Risk Differences")
print("=" * 80)

h2_result = analyze_zipcode_risk(df)
if 'error' not in h2_result:
    print(format_hypothesis_result(h2_result))
    print("\n" + "="*80)
    print("BUSINESS INTERPRETATION:")
    print("="*80)
    print(generate_business_interpretation(h2_result))
else:
    print(f"Note: {h2_result['error']}")
    print("Using Province as proxy for geographic variation...")

## Section 4: Hypothesis 2 - ZipCode Risk Differences

**H₀: There are no risk differences between zip codes.**

Test claim frequency differences between geographic segments using chi-squared test.

In [4]:
print("=" * 80)
print("HYPOTHESIS 1: Province Risk Differences")
print("=" * 80)

h1_result = analyze_province_risk(df)
print(format_hypothesis_result(h1_result))

print("\n" + "="*80)
print("BUSINESS INTERPRETATION:")
print("="*80)
print(generate_business_interpretation(h1_result))

HYPOTHESIS 1: Province Risk Differences


INFO:src.hypothesis_tests:Province Risk: Quebec vs Manitoba: χ² = 0.0000, p = 1.0000


Province Risk: Quebec vs Manitoba
  Test Type: Chi-Squared
  Statistic: 0.0000
  P-value: 1.000000
  Effect Size: 0.0000
  Group A Mean: 1.0000
  Group B Mean: 1.0000
  Decision: ❌ FAIL TO REJECT H₀

BUSINESS INTERPRETATION:
No statistically significant difference found for Province Risk: Quebec vs Manitoba (p = 1.0000). Current segmentation strategy for this factor may be sufficient.


## Section 3: Hypothesis 1 - Province Risk Differences

**H₀: There are no risk differences across provinces.**

Test claim frequency differences between provinces using chi-squared test.

In [3]:
# Calculate loss metrics
df = calculate_loss_metrics(df)

# Ensure required columns exist
if 'HasClaim' not in df.columns:
    df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

if 'Margin' not in df.columns:
    df['Margin'] = df['TotalPremium'] - df['TotalClaims']

if 'LossRatio' not in df.columns:
    df['LossRatio'] = df['TotalClaims'] / df['TotalPremium'].replace(0, np.nan)

# Display summary statistics
print("Risk Metrics Summary:")
print(f"Claim Frequency: {df['HasClaim'].mean():.2%}")
print(f"Average Margin: R{df['Margin'].mean():.2f}")
print(f"Median Margin: R{df['Margin'].median():.2f}")
print(f"\nClaims greater than 0:")
print(f"  Count: {(df['TotalClaims'] > 0).sum()}")
print(f"  Mean Severity (when claim > 0): R{df[df['TotalClaims'] > 0]['TotalClaims'].mean():.2f}")
print(f"  Median Severity (when claim > 0): R{df[df['TotalClaims'] > 0]['TotalClaims'].median():.2f}")

Risk Metrics Summary:
Claim Frequency: 100.00%
Average Margin: R539.84
Median Margin: R542.16

Claims greater than 0:
  Count: 4801
  Mean Severity (when claim > 0): R1259.43
  Median Severity (when claim > 0): R1268.30


## Section 2: Calculate Risk Metrics and Adjust Analysis

Engineer derived features for hypothesis testing.

**Note**: The analysis reveals that claim frequency is 100% in the cleaned dataset (all policies have claims). Therefore:
- H₁ (Province Risk) and H₄ (Gender Risk): Test **Claim Severity** (average TotalClaims) instead of Claim Frequency
- H₂ (ZipCode Risk): Test **Claim Severity** instead of Claim Frequency  
- H₃ (ZipCode Margin): Test **Margin** (TotalPremium - TotalClaims) as planned

In [2]:
# Load cleaned data
df = load_data('../data/insurance_data_cleaned.csv')
print(f"Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\nData shape: {df.shape}")
print(f"Date range: {df['TransactionMonth'].min()} to {df['TransactionMonth'].max()}")
df.head()

INFO:src.data_loader:Loaded 4801 records from ../data/insurance_data_cleaned.csv
INFO:src.data_loader:Validation complete: 4801 records


Dataset loaded: 4801 records, 9 columns

Data shape: (4801, 9)
Date range: 2020-01-01 00:00:00 to 2024-12-01 00:00:00


,PolicyID,TransactionMonth,Province,VehicleType,Gender,Age,DrivingExperience,TotalPremium,TotalClaims
0,1303,2020-01-01,Manitoba,Sedan,F,48,28,2358.464522,876.590204
1,4931,2020-01-01,Alberta,Van,F,68,26,1343.251245,1704.362465
2,3353,2020-01-01,British Columbia,Sedan,M,22,27,857.923553,378.100740
3,1484,2020-01-01,Alberta,Van,M,43,2,2085.717287,794.150019
4,1119,2020-01-01,Quebec,Sedan,M,43,5,1799.446216,950.662197


## Section 1: Data Loading and Preparation

Load the cleaned dataset and calculate key risk metrics.

In [1]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from src.data_loader import load_data
from src.eda_utils import calculate_loss_metrics
from src.hypothesis_tests import (
    analyze_province_risk,
    analyze_zipcode_risk,
    analyze_zipcode_margin,
    analyze_gender_risk,
    independent_t_test,
    format_hypothesis_result,
    generate_business_interpretation
)

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# A/B Hypothesis Testing for Insurance Risk Analysis

Statistical validation of key hypotheses about risk drivers to form the evidence base for ACIS's new segmentation and pricing strategy.

**KPIs Used:**
- **Claim Frequency**: Proportion of policies with at least one claim (binary outcome)
- **Claim Severity**: Average claim amount given a claim occurred (numerical)
- **Margin**: Profit metric defined as TotalPremium - TotalClaims (numerical)